# Section 04 lab — Aggregations and Spark SQL

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Count rows in each group

Start with the simplest aggregation: one count for each distinct value.


### Step 1 — Create status rows

Repeated statuses form visible groups.


In [ ]:
orders = spark.createDataFrame([('O1', 'COMPLETED'), ('O2', 'CANCELLED'), ('O3', 'COMPLETED'), ('O4', 'RETURNED'), ('O5', 'COMPLETED')], ['order_id', 'status'])
orders.show()


### Step 2 — Group by status

groupBy changes the result from order rows to status groups.


In [ ]:
grouped = orders.groupBy('status')
grouped


### Step 3 — Count each group

count produces one row per status.


In [ ]:
status_counts = grouped.count().orderBy('status')
status_counts.show()
assert status_counts.count() == 3


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
orders.createOrReplaceTempView('section04_orders')
sql_result = spark.sql('SELECT status, COUNT(*) AS count FROM section04_orders GROUP BY status ORDER BY status')
sql_result.show()
assert sql_result.collect() == status_counts.collect()


## Exercise 2 — Calculate several measures

Use agg to describe more than one measure for every category.


### Step 1 — Create sales lines

Each line has a category, quantity and revenue.


In [ ]:
sales = spark.createDataFrame([('Accessories', 2, 1000.0), ('Accessories', 1, 500.0), ('Audio', 1, 1200.0), ('Audio', 2, 2400.0)], ['category', 'quantity', 'revenue'])


### Step 2 — Define aggregate expressions

Each alias names one output measure.


In [ ]:
from pyspark.sql import functions as F
summary = sales.groupBy('category').agg(F.count('*').alias('lines'), F.sum('quantity').alias('units'), F.avg('revenue').alias('average_revenue'), F.sum('revenue').alias('revenue'))


### Step 3 — Inspect the category result

There should be one row for each category.


In [ ]:
summary.orderBy('category').show()
assert summary.count() == 2


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
sales.createOrReplaceTempView('section04_sales_measures')
sql_result = spark.sql("""
SELECT category, COUNT(*) AS lines, SUM(quantity) AS units,
       AVG(revenue) AS average_revenue, SUM(revenue) AS revenue
FROM section04_sales_measures
GROUP BY category
ORDER BY category
""")
sql_result.show()
assert sql_result.count() == summary.count()


## Exercise 3 — Group by two columns

Increase the level of detail from category to date-and-category.


### Step 1 — Create dated sales

Both dates contain more than one category.


In [ ]:
sales = spark.createDataFrame([('2026-10-01', 'Accessories', 500.0), ('2026-10-01', 'Audio', 1200.0), ('2026-10-02', 'Accessories', 750.0), ('2026-10-02', 'Audio', 2400.0)], ['order_date', 'category', 'revenue'])


### Step 2 — Group by date and category

Every unique combination becomes one group.


In [ ]:
from pyspark.sql import functions as F
daily_category = sales.groupBy('order_date', 'category').agg(F.sum('revenue').alias('revenue'))


### Step 3 — Sort the grouped output

The familiar orderBy operation makes the groups easy to read.


In [ ]:
daily_category.orderBy('order_date', 'category').show()
assert daily_category.count() == 4


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
sales.createOrReplaceTempView('section04_daily_sales')
sql_result = spark.sql("""
SELECT order_date, category, SUM(revenue) AS revenue
FROM section04_daily_sales
GROUP BY order_date, category
ORDER BY order_date, category
""")
sql_result.show()
assert sql_result.collect() == daily_category.orderBy("order_date", "category").collect()


## Exercise 4 — Rank aggregated results

Apply orderBy after the aggregation rather than to the original rows.


### Step 1 — Create sales with repeated categories

Repeated rows make category totals different.


In [ ]:
sales = spark.createDataFrame([('Accessories', 500.0), ('Audio', 1200.0), ('Accessories', 750.0), ('Audio', 2400.0), ('Stationery', 150.0)], ['category', 'revenue'])


### Step 2 — Calculate category totals

The grouped DataFrame has one row per category.


In [ ]:
from pyspark.sql import functions as F
totals = sales.groupBy('category').agg(F.sum('revenue').alias('revenue'))


### Step 3 — Order by the calculated measure

The largest total appears first.


In [ ]:
ranked = totals.orderBy(F.desc('revenue'), F.asc('category'))
ranked.show()
assert ranked.first()['category'] == 'Audio'


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
sales.createOrReplaceTempView('section04_rank_source')
sql_result = spark.sql("""
SELECT category, SUM(revenue) AS revenue
FROM section04_rank_source
GROUP BY category
ORDER BY revenue DESC, category ASC
""")
sql_result.show()
assert sql_result.first()['category'] == ranked.first()['category']


## Exercise 5 — Express an aggregation with Spark SQL

Translate operations already used in the DataFrame API into SQL.


### Step 1 — Create the sales DataFrame

The input is identical in shape to the previous exercise.


In [ ]:
sales = spark.createDataFrame([('Accessories', 500.0), ('Audio', 1200.0), ('Accessories', 750.0), ('Audio', 2400.0), ('Stationery', 150.0)], ['category', 'revenue'])


### Step 2 — Register a temporary view

The view gives the DataFrame a name for SQL queries.


In [ ]:
sales.createOrReplaceTempView('section04_sales')


### Step 3 — Group and order with SQL

The query follows the same sequence as groupBy, agg and orderBy.


In [ ]:
result = spark.sql("""
SELECT category, COUNT(*) AS lines, SUM(revenue) AS revenue
FROM section04_sales
GROUP BY category
ORDER BY revenue DESC, category ASC
""")
result.show()
assert result.first()['category'] == 'Audio'


### DataFrame API comparison

The exercise already used literal SQL; reproduce the same result with groupBy, agg and orderBy.


In [ ]:
from pyspark.sql import functions as F
dataframe_result = (sales.groupBy('category').agg(F.count('*').alias('lines'), F.sum('revenue').alias('revenue')).orderBy(F.desc('revenue'), F.asc('category')))
dataframe_result.show()
assert dataframe_result.collect() == result.collect()
